In [ ]:
import os
import json
import re
from typing import List, Optional, Dict, Any
from enum import Enum
from dotenv import load_dotenv
from pydantic import BaseModel, Field, validator, model_validator
from typing import List, Optional, Dict, Any
from typing_extensions import TypedDict
from enum import Enum
from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama
from langchain_openai import ChatOpenAI
import openpyxl
import time
from langchain_core.messages import SystemMessage, HumanMessage
from functools import partial
from langgraph.graph import StateGraph, END
import openpyxl
from dataclasses import dataclass, field
from typing import List, Optional
import numpy as np
from sentence_transformers import SentenceTransformer, util
from langchain_core.messages import SystemMessage, HumanMessage




In [38]:
temp = 0.0
max_intentos = 3
modelo_gpt = "gpt-4o-mini"
llm_gpt = ChatOpenAI(model=modelo_gpt,temperature=temp,api_key=OPENAI_API_KEY)
llm_juez = llm_gpt

## Definición de las clases y Normalización: ##

In [39]:
class clasf_proyecto(str, Enum):
    apto = "apartamento"
    oficina = "oficina"
    consultorio = "consultorio"
    tienda = "tienda"
    restaurante = "restaurante"

class estado_obra(str, Enum):
    obra_civil_remodelacion = "remodelacion o obra civil"
    obra_nueva = "obra nueva"

class capitulo(str, Enum):
    preliminares = "CAP01_PRELIMINARES"
    obra_civil = "CAP02_OBRA_CIVIL_ESTRUCTURA"
    panetes_enchapes = "CAP03_PANETES_ENCHAPES_CIELORASOS"
    pisos = "CAP04_PISOS"
    carpinteria = "CAP05_CARPINTERIA_MUEBLES"
    electrico = "CAP06_ELECTRICO_DATOS"
    hidrosanitario = "CAP07_HIDROSANITARIO"
    aire_acond = "CAP08_AIRE_ACONDICIONADO"
    seguridad = "CAP09_SEGURIDAD_DATOS"
    adicionales = "CAP10_ADICIONALES"

class unidad_medida(str, Enum):
    m2 = "m2"
    ml = "ml"
    un = "un"
    m3 = "m3"
    gl = "gl"
    kg = "kg"

class suministro(str, Enum):
    cliente = "cliente"
    contratista = "contratista"
    por_definir = "por_definir"

class origen_desc(str, Enum):
    explicita = "explicita"
    inferida = "inferida"

Normalización de la data:

In [40]:
# Normalizacion de la unidad de medida
norm_unidad = {
    "metros cuadrados": "m2", "metro cuadrado": "m2",
    "m²": "m2", "mts2": "m2", "mt2": "m2", "M2": "m2",
    "metros lineales": "ml", "metro lineal": "ml",
    "m.l.": "ml", "mts": "ml", "Ml": "ml", "ML": "ml", "m": "ml",
    "metros cubicos": "m3", "metro cubico": "m3",
    "m³": "m3", "M3": "m3",
    "unidad": "un", "unidades": "un", "und": "un",
    "ud": "un", "UND": "un", "Un": "un", "u": "un", "U": "un",
    "global": "gl", "gb": "gl", "GLB": "gl",
    "Global": "gl", "gl": "gl", "GL": "gl", "glb": "gl",
    "glbl": "gl", "Glbl": "gl", "GLBL": "gl",
    "Kg": "kg", "KG": "kg", "kgs": "kg",
    "kilogramo": "kg", "kilogramos": "kg",
    "mm": "un", "MM": "un", "cm": "un", "CM": "un",
    "sem": "un", "SEM": "un",
    "dia": "un", "día": "un", "Día": "un", "DIA": "un", "DÍA": "un",
}

# Normalización de los capitulos de las descripciones de actividades
norm_capitulos = {
    "preliminares": "CAP01_PRELIMINARES",
    "preliminares y desmonte": "CAP01_PRELIMINARES",
    "desmonte y desconexiones": "CAP01_PRELIMINARES",
    "desmontes y desconexiones": "CAP01_PRELIMINARES",
    "mamposteria": "CAP02_OBRA_CIVIL_ESTRUCTURA",
    "muros": "CAP02_OBRA_CIVIL_ESTRUCTURA",
    "muros mamposterias panetes enchapes": "CAP02_OBRA_CIVIL_ESTRUCTURA",
    "drywall": "CAP02_OBRA_CIVIL_ESTRUCTURA",
    "drywall y super board": "CAP02_OBRA_CIVIL_ESTRUCTURA",
    "drywall y super board pintura": "CAP02_OBRA_CIVIL_ESTRUCTURA",
    "muros drywall superboard": "CAP02_OBRA_CIVIL_ESTRUCTURA",
    "estructura metalica": "CAP02_OBRA_CIVIL_ESTRUCTURA",
    "estructura": "CAP02_OBRA_CIVIL_ESTRUCTURA",
    "concreto": "CAP02_OBRA_CIVIL_ESTRUCTURA",
    "refuerzos": "CAP02_OBRA_CIVIL_ESTRUCTURA",
    "cubierta": "CAP02_OBRA_CIVIL_ESTRUCTURA",
    "tendido de cubierta": "CAP02_OBRA_CIVIL_ESTRUCTURA",
    "techos": "CAP02_OBRA_CIVIL_ESTRUCTURA",
    "muros y techo": "CAP02_OBRA_CIVIL_ESTRUCTURA",
    "panetes enchapes cielorasos": "CAP03_PANETES_ENCHAPES_CIELORASOS",
    "pañetes enchapes cielorasos": "CAP03_PANETES_ENCHAPES_CIELORASOS",
    "enchapes": "CAP03_PANETES_ENCHAPES_CIELORASOS",
    "enchapes y acabados": "CAP03_PANETES_ENCHAPES_CIELORASOS",
    "cielorasos": "CAP03_PANETES_ENCHAPES_CIELORASOS",
    "recubrimientos en madera": "CAP03_PANETES_ENCHAPES_CIELORASOS",
    "pintura general": "CAP03_PANETES_ENCHAPES_CIELORASOS",
    "pintura": "CAP03_PANETES_ENCHAPES_CIELORASOS",
    "piso": "CAP04_PISOS",
    "pisos": "CAP04_PISOS",
    "pisos y paredes": "CAP04_PISOS",
    "piso microcemento": "CAP04_PISOS",
    "guarda escobas": "CAP04_PISOS",
    "carpinteria": "CAP05_CARPINTERIA_MUEBLES",
    "carpinteria en madera": "CAP05_CARPINTERIA_MUEBLES",
    "carpinterias aluminio cristales": "CAP05_CARPINTERIA_MUEBLES",
    "mesones": "CAP05_CARPINTERIA_MUEBLES",
    "muebles": "CAP05_CARPINTERIA_MUEBLES",
    "mobiliario fijo": "CAP05_CARPINTERIA_MUEBLES",
    "acero inoxidable": "CAP05_CARPINTERIA_MUEBLES",
    "fachada": "CAP05_CARPINTERIA_MUEBLES",
    "instalacion electrica y datos": "CAP06_ELECTRICO_DATOS",
    "instalaciones electricas y datos": "CAP06_ELECTRICO_DATOS",
    "instalacion electrica  y datos": "CAP06_ELECTRICO_DATOS",
    "electrico": "CAP06_ELECTRICO_DATOS",
    "redes electricas": "CAP06_ELECTRICO_DATOS",
    "red normal": "CAP06_ELECTRICO_DATOS",
    "tableros y acometidas": "CAP06_ELECTRICO_DATOS",
    "tablero electrico y rack": "CAP06_ELECTRICO_DATOS",
    "tableros electricos": "CAP06_ELECTRICO_DATOS",
    "iluminacion": "CAP06_ELECTRICO_DATOS",
    "iluminacion especial": "CAP06_ELECTRICO_DATOS",
    "instalacion de datos": "CAP06_ELECTRICO_DATOS",
    "sistema de comunicaciones": "CAP06_ELECTRICO_DATOS",
    "instalaciones datos sonido cctv": "CAP06_ELECTRICO_DATOS",
    "cableado de acometidas": "CAP06_ELECTRICO_DATOS",
    "diseño electrico y datos": "CAP06_ELECTRICO_DATOS",
    "especiales": "CAP06_ELECTRICO_DATOS",
    "instalaciones hidrosanitarias": "CAP07_HIDROSANITARIO",
    "hidraulicos barra": "CAP07_HIDROSANITARIO",
    "hidrosanitarios": "CAP07_HIDROSANITARIO",
    "hidrosanitario y gas": "CAP07_HIDROSANITARIO",
    "redes hidrosanitarias": "CAP07_HIDROSANITARIO",
    "instalaciones hidraulicas": "CAP07_HIDROSANITARIO",
    "instalaciones sanitarias": "CAP07_HIDROSANITARIO",
    "aparatos sanitarios griferias y accesorios": "CAP07_HIDROSANITARIO",
    "aparatos sanitarios griferias accesorios": "CAP07_HIDROSANITARIO",
    "equipos y accesorios de bano": "CAP07_HIDROSANITARIO",
    "diseño hidrosanitario": "CAP07_HIDROSANITARIO",
    "aire acondicionado": "CAP08_AIRE_ACONDICIONADO",
    "ductos": "CAP08_AIRE_ACONDICIONADO",
    "red de detencion de incendios": "CAP09_SEGURIDAD_DATOS",
    "rci": "CAP09_SEGURIDAD_DATOS",
    "red contra incendios": "CAP09_SEGURIDAD_DATOS",
    "red de incendio": "CAP09_SEGURIDAD_DATOS",
    "diseño rci deteccion y especiales": "CAP09_SEGURIDAD_DATOS",
    "diseño deteccion": "CAP09_SEGURIDAD_DATOS",
    "cctv": "CAP09_SEGURIDAD_DATOS",
    "control acceso": "CAP09_SEGURIDAD_DATOS",
    "sistema intrusion": "CAP09_SEGURIDAD_DATOS",
    "cctv y datos": "CAP09_SEGURIDAD_DATOS",
    "instalaciones de cctv": "CAP09_SEGURIDAD_DATOS",
    "protecciones": "CAP09_SEGURIDAD_DATOS",
    "otros": "CAP10_ADICIONALES",
    "otros items": "CAP10_ADICIONALES",
    "adicionales": "CAP10_ADICIONALES",
    "ornamentacion": "CAP10_ADICIONALES",
    "bodega": "CAP10_ADICIONALES",
    "bodega interna": "CAP10_ADICIONALES",
    "equipos especiales": "CAP10_ADICIONALES",
    "equipos": "CAP10_ADICIONALES",
    "aseo y entrega": "CAP10_ADICIONALES",
    "aseo y entrega de obra": "CAP10_ADICIONALES",
    "gastos administrativos": "CAP10_ADICIONALES",
    "personal exigido": "CAP10_ADICIONALES",
}

#Normalizacion estado
norm_estado = {
    "obra civil o remodelacion": "remodelacion o obra civil",
    "remodelacion o obra civil": "remodelacion o obra civil",
    "remodelacion": "remodelacion o obra civil",
    "adecuacion": "remodelacion o obra civil",
    "adecuación": "remodelacion o obra civil",
    "reforma": "remodelacion o obra civil",
    "intervencion": "remodelacion o obra civil",
    "intervención": "remodelacion o obra civil",
    "renovacion": "remodelacion o obra civil",
    "obra civil": "remodelacion o obra civil",
    "obra nueva": "obra nueva",
    "obra gris": "obra nueva",
    "gris": "obra nueva",
    "entregado en gris": "obra nueva",
    "sin acabados": "obra nueva",
    "construccion nueva": "obra nueva",
    "construccion": "obra nueva",
}



**Estandarización capitulos del ground thruth:** 
Esto se hace dado que cada proyeccto cuenta con unos capitulos y agrupaciones diferentes por lo que se necesita generar una estructura. 

In [41]:
def norm_texto(v: str) -> str:
    v = v.lower().strip()
    for orig, dest in {"á":"a","é":"e","í":"i","ó":"o","ú":"u","ñ":"n","-":" ","_":" "}.items():
        v = v.replace(orig, dest)
    return " ".join(v.split())

def norm_capitulos_fn(nombre):
    return norm_capitulos.get(norm_texto(nombre), "NO_MAPEADO")

## PyDantic ##

**Modelo PyDantic:** Se estructura el modelo generando cada una de las reglas.

actividad: Se estructura las actividades

In [42]:
class actividad(BaseModel):
    capitulo: capitulo
    descripcion: str
    referencia: Optional[str] = None
    unidad: unidad_medida
    cantidad: float
    suministro: suministro        
    origen: origen_desc = origen_desc.explicita
    observacion: Optional[str] = None

    @validator("unidad", pre=True)
    def normalizar_unidad(cls, v):
        if isinstance(v, str):
            val = v.strip()
            if val in norm_unidad:
                return norm_unidad[val]
            if val in [e.value for e in unidad_medida]:
                return val
            raise ValueError(f"unidad '{v}' no válida. usar: m2, ml, un, m3, glb")
        return v
 
    @validator("descripcion")
    def validar_descripcion(cls, v):
        v = v.strip()
        if len(v) < 5:
            raise ValueError("descripcion muy corta")
        return v
 
    @validator("referencia")
    def validar_referencia(cls, v):
        if v is not None:
            v = v.strip().upper()
            if len(v) < 3:
                raise ValueError("referencia muy corta")
        return v
 
    @validator("cantidad")
    def validar_cantidad(cls, v):
        if v is None:
            raise ValueError("cantidad obligatoria")
        if v <= 0:
            raise ValueError("cantidad debe ser mayor a cero")
        return round(v, 2)

C:\Users\Dell\AppData\Local\Temp\ipykernel_34612\421033201.py:11: PydanticDeprecatedSince20: Pydantic V1 style `@validator` validators are deprecated. You should migrate to Pydantic V2 style `@field_validator` validators, see the migration guide for more details. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  @validator("unidad", pre=True)
C:\Users\Dell\AppData\Local\Temp\ipykernel_34612\421033201.py:22: PydanticDeprecatedSince20: Pydantic V1 style `@validator` validators are deprecated. You should migrate to Pydantic V2 style `@field_validator` validators, see the migration guide for more details. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  @validator("descripcion")
C:\Users\Dell\AppData\Local\Temp\ipykernel_34612\421033201.py:29: PydanticDeprecatedSince20: Pydantic V1 style `@validator` validators are deprecated. 

In [43]:
class espacio(BaseModel):
    nombre: str
    area_m2: Optional[float] = None
    actividades: List[actividad] = Field(default_factory=list)
 
    @validator("nombre")
    def normalizar_nombre(cls, valor):
       return norm_texto(valor).replace(" ", "_")
    
    @validator("area_m2")
    def area_valida(cls, valor):
        if valor is not None and valor <= 0:
            raise ValueError("area debe ser mayor a cero")
        return round(valor, 2) if valor else valor
 
    @validator("actividades")
    def act_valida(cls, actividades, values):
        area = values.get("area_m2")
        if area:
            for act in actividades:
                if act.unidad == unidad_medida.m2 and act.cantidad > area * 5:
                    raise ValueError(
                        f"cantidad sospechosa: {act.descripcion} "
                        f"tiene {act.cantidad}m2 en espacio de {area}m2"
                    )
        return actividades

C:\Users\Dell\AppData\Local\Temp\ipykernel_34612\2073103792.py:6: PydanticDeprecatedSince20: Pydantic V1 style `@validator` validators are deprecated. You should migrate to Pydantic V2 style `@field_validator` validators, see the migration guide for more details. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  @validator("nombre")
C:\Users\Dell\AppData\Local\Temp\ipykernel_34612\2073103792.py:10: PydanticDeprecatedSince20: Pydantic V1 style `@validator` validators are deprecated. You should migrate to Pydantic V2 style `@field_validator` validators, see the migration guide for more details. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  @validator("area_m2")
C:\Users\Dell\AppData\Local\Temp\ipykernel_34612\2073103792.py:16: PydanticDeprecatedSince20: Pydantic V1 style `@validator` validators are deprecated. You should m

Proyecto: Genera el output final con la plantilla APU

In [44]:
class proyecto(BaseModel):
    tipo:    clasf_proyecto
    estado:           estado_obra
    area_total:       Optional[float] = None
    altura_entrepiso: float = 2.40
    espacios:         List[espacio] = Field(default_factory=list)
    alertas:          List[str]     = Field(default_factory=list)
 
    @validator("estado", pre=True)
    def normalizar_estado(cls, v):
        return norm_estado.get(norm_texto(str(v)), str(v))
 
    @validator("area_total")
    def validar_area_total(cls, v):
        if v is not None and v <= 0:
            raise ValueError("area total no es correcta")
        return round(v, 2) if v else v
 
    @validator("altura_entrepiso")
    def validar_altura(cls, v):
        if v <= 0 or v > 6.0:
            raise ValueError("altura de entrepiso no es correcta")
        return round(v, 2)
 
    @validator("espacios")
    def validar_espacios(cls, v):
        if not v:
            return v
        nombres = [e.nombre for e in v]
        if len(nombres) != len(set(nombres)):
            dups = {n for n in nombres if nombres.count(n) > 1}
            raise ValueError(f"espacios duplicados: {dups}")
        return v
 
    @model_validator(mode="after")
    def validaciones_cruzadas(self):
        # demoliciones
        if self.estado == estado_obra.obra_nueva:
            for esp in self.espacios:
                caps = [a.capitulo for a in esp.actividades]
                if capitulo.preliminares in caps:
                    self.alertas.append(
                        f"alerta: {esp.nombre} tiene demoliciones "
                        f"pero el estado es obra nueva"
                    )
        # calcular area si no viene
        if self.area_total is None:
            areas = [e.area_m2 for e in self.espacios if e.area_m2]
            if areas:
                self.area_total = round(sum(areas), 2)
                self.alertas.append(
                    f"supuesto: area total = {self.area_total}m2"
                )
        # diferencia area > 10%
        if self.area_total:
            areas = [e.area_m2 for e in self.espacios if e.area_m2]
            if areas:
                suma = round(sum(areas), 2)
                if abs(self.area_total - suma) > self.area_total * 0.10:
                    self.alertas.append(
                        f"alerta: area declarada ({self.area_total}m2) "
                        f"difiere >10% de suma espacios ({suma}m2)"
                    )
        # espacios sin area
        for esp in self.espacios:
            if esp.area_m2 is None:
                self.alertas.append(
                    f"pregunta: cuantos m2 tiene {esp.nombre}?"
                )
        return self
 
    @property
    def area_calculada(self):
        return round(sum(e.area_m2 for e in self.espacios if e.area_m2), 2)
 
    @property
    def tiene_preguntas(self):
        return any("pregunta:" in a for a in self.alertas)
 
    @property
    def preguntas_pendientes(self):
        return [a for a in self.alertas if "pregunta:" in a]
 
    @property
    def supuestos(self):
        return [a for a in self.alertas if "supuesto:" in a]
 
    @property
    def inconsistencias(self):
        return [a for a in self.alertas if "alerta:" in a]
 
    @property
    def actividades_planas(self):
        return [
            {
                "espacio": esp.nombre,
                "capitulo": act.capitulo.value,
                "descripcion": act.descripcion,
                "referencia": act.referencia,
                "unidad": act.unidad.value,
                "cantidad": act.cantidad,
                "suministro": act.suministro.value,
                "origen": act.origen.value,
            }
            for esp in self.espacios
            for act in esp.actividades
        ]
 
    @property
    def actividades_cliente(self):
        return [
            {
                "espacio": esp.nombre,
                "actividad": act.descripcion,
                "referencia": act.referencia,
                "cantidad": act.cantidad,
                "unidad": act.unidad.value,
            }
            for esp in self.espacios
            for act in esp.actividades
            if act.suministro == suministro.cliente
        ]
 
    @property
    def resumen_capitulos(self):
        res = {}
        for esp in self.espacios:
            for act in esp.actividades:
                cap = act.capitulo.value
                if cap not in res:
                    res[cap] = {"n": 0, "espacios": []}
                res[cap]["n"] += 1
                if esp.nombre not in res[cap]["espacios"]:
                    res[cap]["espacios"].append(esp.nombre)
        return res
 
    def planilla_texto(self):
        lineas = [
            f"proyecto: {self.tipo.value.upper()} | {self.estado.value.upper()}",
            f"area: {self.area_total}m2 | altura: {self.altura_entrepiso}m",
            "=" * 60,
        ]
        for esp in self.espacios:
            lineas.append(f"\n{esp.nombre.upper()} ({esp.area_m2}m2)")
            cap_actual = None
            for act in esp.actividades:
                if act.capitulo != cap_actual:
                    lineas.append(f"  [{act.capitulo.value}]")
                    cap_actual = act.capitulo
                tag = {"cliente": "cliente", "contratista": "contratista",
                       "por_definir": "sin definir"}[act.suministro.value]
                lineas.append(
                    f"    {act.descripcion:<45} "
                    f"{act.cantidad:>7.2f} {act.unidad.value:<4} [{tag}]"
                )
        if self.alertas:
            lineas.append("alertas:")
            lineas.extend(f"  {a}" for a in self.alertas)
        return "\n".join(lineas)

C:\Users\Dell\AppData\Local\Temp\ipykernel_34612\2343993277.py:9: PydanticDeprecatedSince20: Pydantic V1 style `@validator` validators are deprecated. You should migrate to Pydantic V2 style `@field_validator` validators, see the migration guide for more details. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  @validator("estado", pre=True)
C:\Users\Dell\AppData\Local\Temp\ipykernel_34612\2343993277.py:13: PydanticDeprecatedSince20: Pydantic V1 style `@validator` validators are deprecated. You should migrate to Pydantic V2 style `@field_validator` validators, see the migration guide for more details. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  @validator("area_total")
C:\Users\Dell\AppData\Local\Temp\ipykernel_34612\2343993277.py:19: PydanticDeprecatedSince20: Pydantic V1 style `@validator` validators are deprecated.

## Agentstate ##

Evaluación den agente:

In [45]:
class modo_ejecucuion(str, Enum):
    eval = "evaluacion"
    prod = "produccion"

In [46]:
class AgentState(TypedDict):
    #Outup
    texto: str
    modo: str                    
    caso_id: Optional[str]         
    #Procesamiento
    output_raw: Optional[Dict[str, Any]]
    proyecto: Optional[Dict[str, Any]]
    schema_ok: bool
    intentos: int
    error_pydantic: Optional[str]
    bloqueado: bool
    #Observabilidad
    trajectory: List[Dict[str, Any]]
    #Metricas
    modelo_llm: str
    latencia_ms: Optional[float]
 
def init_state(texto: str, modo: str = modo_ejecucuion.eval, caso_id: Optional[str] = None, modelo_llm: str = "gpt-4o-mini"): 
    return AgentState(
        texto= texto,
        modo= modo,
        caso_id = caso_id,
        output_raw = None,
        proyecto = None,
        schema_ok = False,
        intentos = 0,
        error_pydantic = None,
        bloqueado = False,
        trajectory = [],
        modelo_llm = modelo_llm,
        latencia_ms = None,
    )
 
 
def registrar_trajectory(state, componente, decision, resultado, detalle):
    state["trajectory"].append({
        "componente": componente,
        "decision": decision,
        "resultado": resultado,
        "detalle": detalle,
    })
 
 

### Prompt Enginere 

**Prompt con Brief:**

In [47]:
SYSTEM_PROMPT = """
Eres un interventor de obra con 15 años de experiencia en proyectos de adecuación,
remodelación y construcción de espacios comerciales y de oficinas en Colombia.

La funcion que debes realizar es leer la descripción de un proyecto constructivo y generar una planilla
de estimación de costos APU organizada por capitulos y actividades.

Debes hacer dos cosas:
1. EXTRAER las actividades que el contratista describe explícitamente en el texto.
2. INFERIR las actividades que aplican según el tipo de proyecto y estado de la obra,
   aunque el contratista no las mencione.


Formato de la respuesta:
Responde unicamente con un JSON válido con esta estructura exacta:

{
  "tipo": "<apartamento|oficina|clinica|consultorio|tienda|restaurante>",
  "estado": "<remodelacion o obra civil|obra nueva>",
  "area_total": <número o null>,
  "altura_entrepiso": <número, default 2.40>,
  "espacios": [
    {
      "nombre": "<nombre_sin_tildes_ni_espacios>",
      "area_m2": <número o null>,
      "actividades": [
        {
          "capitulo": "<CAP01_PRELIMINARES|CAP02_OBRA_CIVIL_ESTRUCTURA|CAP03_PANETES_ENCHAPES_CIELORASOS|CAP04_PISOS|CAP05_CARPINTERIA_MUEBLES|CAP06_ELECTRICO_DATOS|CAP07_HIDROSANITARIO|CAP08_AIRE_ACONDICIONADO|CAP09_SEGURIDAD_DATOS|CAP10_ADICIONALES>",
          "descripcion": "<descripción técnica de la actividad>",
          "referencia": "<referencia del material o null>",
          "unidad": "<m2|ml|un|m3|gl>",
          "cantidad": <número mayor a cero>,
          "suministro": "<contratista|cliente|por_definir>",
          "origen": "<explicita|inferida>",
          "observacion": "<supuesto aplicado o null>"
        }
      ]
    }
  ],
  "alertas": []
}

Ejemplos:

Ejemplo 1: el tipo de proyecto es un consultorio, con estado de la obra en remodelacion y un area total de 108 m2:
Este es el detalle y completitud esperado para un proyecto de obra civil o remodelacion.

{
  "tipo": "consultorio",
  "estado": "remodelacion o obra civil",
  "area_total": 108,
  "altura_entrepiso": 2.40,
  "espacios": [
    {
      "nombre": "area_general",
      "area_m2": 108,
      "actividades": [
        {"capitulo":"CAP01_PRELIMINARES","descripcion":"Demolicion de muros de drywall","referencia":null,"unidad":"m2","cantidad":36,"suministro":"contratista","origen":"explicita","observacion":null},
        {"capitulo":"CAP01_PRELIMINARES","descripcion":"Desmonte de piso existente","referencia":null,"unidad":"m2","cantidad":120,"suministro":"contratista","origen":"inferida","observacion":"supuesto: desmonte previo a piso nuevo"},
        {"capitulo":"CAP01_PRELIMINARES","descripcion":"Desmonte de cieloraso existente","referencia":null,"unidad":"m2","cantidad":120,"suministro":"contratista","origen":"inferida","observacion":null},
        {"capitulo":"CAP01_PRELIMINARES","descripcion":"Desmonte de redes electricas y de datos existentes","referencia":null,"unidad":"gl","cantidad":1,"suministro":"contratista","origen":"inferida","observacion":null},
        {"capitulo":"CAP01_PRELIMINARES","descripcion":"Regata en muros y pisos","referencia":null,"unidad":"ml","cantidad":28,"suministro":"contratista","origen":"inferida","observacion":null},
        {"capitulo":"CAP01_PRELIMINARES","descripcion":"Material de proteccion","referencia":null,"unidad":"gl","cantidad":1,"suministro":"contratista","origen":"inferida","observacion":null},
        {"capitulo":"CAP01_PRELIMINARES","descripcion":"Retiro de escombros","referencia":null,"unidad":"gl","cantidad":1,"suministro":"contratista","origen":"inferida","observacion":null},
        {"capitulo":"CAP02_OBRA_CIVIL_ESTRUCTURA","descripcion":"Muro drywall doble cara con frescasa","referencia":null,"unidad":"m2","cantidad":71,"suministro":"contratista","origen":"explicita","observacion":null},
        {"capitulo":"CAP02_OBRA_CIVIL_ESTRUCTURA","descripcion":"Muro superboard","referencia":null,"unidad":"m2","cantidad":25,"suministro":"contratista","origen":"explicita","observacion":null},
        {"capitulo":"CAP02_OBRA_CIVIL_ESTRUCTURA","descripcion":"Dinteles en drywall","referencia":null,"unidad":"ml","cantidad":16,"suministro":"contratista","origen":"inferida","observacion":null},
        {"capitulo":"CAP03_PANETES_ENCHAPES_CIELORASOS","descripcion":"Resanes de regatas en muros y pisos","referencia":null,"unidad":"ml","cantidad":25,"suministro":"contratista","origen":"inferida","observacion":null},
        {"capitulo":"CAP03_PANETES_ENCHAPES_CIELORASOS","descripcion":"Instalacion enchape muro porcelanato Amalfi Inout 100x100 Beige","referencia":"Amalfi Inout 100x100 Beige","unidad":"m2","cantidad":28,"suministro":"contratista","origen":"explicita","observacion":null},
        {"capitulo":"CAP03_PANETES_ENCHAPES_CIELORASOS","descripcion":"Cieloraso en drywall","referencia":null,"unidad":"m2","cantidad":120,"suministro":"contratista","origen":"explicita","observacion":null},
        {"capitulo":"CAP03_PANETES_ENCHAPES_CIELORASOS","descripcion":"Tapas de inspeccion cieloraso 60x60 tipo push","referencia":null,"unidad":"un","cantidad":4,"suministro":"contratista","origen":"inferida","observacion":"supuesto: tapas de inspeccion para cieloraso drywall"},
        {"capitulo":"CAP03_PANETES_ENCHAPES_CIELORASOS","descripcion":"Dilataciones en Z cieloraso","referencia":null,"unidad":"ml","cantidad":48,"suministro":"contratista","origen":"inferida","observacion":null},
        {"capitulo":"CAP03_PANETES_ENCHAPES_CIELORASOS","descripcion":"Pintura viniilica muros","referencia":null,"unidad":"m2","cantidad":184,"suministro":"contratista","origen":"inferida","observacion":null},
        {"capitulo":"CAP03_PANETES_ENCHAPES_CIELORASOS","descripcion":"Estuco muros","referencia":null,"unidad":"m2","cantidad":101,"suministro":"contratista","origen":"inferida","observacion":null},
        {"capitulo":"CAP04_PISOS","descripcion":"Alistado de piso","referencia":null,"unidad":"m2","cantidad":108,"suministro":"contratista","origen":"inferida","observacion":"supuesto: alistado previo a piso nuevo"},
        {"capitulo":"CAP04_PISOS","descripcion":"Instalacion enchape piso porcelanato Neeko Inout 60x120 Blanco","referencia":"Neeko Inout 60x120 Blanco","unidad":"m2","cantidad":108,"suministro":"contratista","origen":"explicita","observacion":null},
        {"capitulo":"CAP04_PISOS","descripcion":"Guardaescoba aglomerado segun referencia","referencia":null,"unidad":"ml","cantidad":42,"suministro":"contratista","origen":"inferida","observacion":null},
        {"capitulo":"CAP05_CARPINTERIA_MUEBLES","descripcion":"Puerta corrediza vidrio laminado 3+3 perfil 1101","referencia":null,"unidad":"un","cantidad":2,"suministro":"contratista","origen":"explicita","observacion":null},
        {"capitulo":"CAP05_CARPINTERIA_MUEBLES","descripcion":"Puerta batiente vidrio laminado 3+3 cerradura llave llave","referencia":null,"unidad":"un","cantidad":4,"suministro":"contratista","origen":"explicita","observacion":null},
        {"capitulo":"CAP06_ELECTRICO_DATOS","descripcion":"Salida para luminaria colgante campana incluye suministro","referencia":null,"unidad":"un","cantidad":6,"suministro":"contratista","origen":"explicita","observacion":null},
        {"capitulo":"CAP06_ELECTRICO_DATOS","descripcion":"Salida para tomacorriente red normal doble Nema 5-15R Leviton","referencia":"Leviton Nema 5-15R","unidad":"un","cantidad":18,"suministro":"contratista","origen":"explicita","observacion":null},
        {"capitulo":"CAP06_ELECTRICO_DATOS","descripcion":"Salida para punto de datos CAT 6A","referencia":null,"unidad":"un","cantidad":12,"suministro":"contratista","origen":"explicita","observacion":null},
        {"capitulo":"CAP06_ELECTRICO_DATOS","descripcion":"Patch panel CAT 6A 24 puertos","referencia":null,"unidad":"un","cantidad":1,"suministro":"contratista","origen":"explicita","observacion":null},
        {"capitulo":"CAP06_ELECTRICO_DATOS","descripcion":"Acondicionamiento tablero electrico existente","referencia":null,"unidad":"gl","cantidad":1,"suministro":"contratista","origen":"explicita","observacion":null}
      ]
    }
  ],
  "alertas": ["pregunta: tiene referencia de enchape de piso?", "supuesto: altura entrepiso 2.40m, confirmar"]
}

Ejemplo 2: el tipo de proyecto es una tienda, con estado en obra nueva y un area total de 80m2:
Este es el detalle y completitud esperado para un proyecto de obra civil o remodelacion.

{
  "tipo": "tienda",
  "estado": "obra nueva",
  "area_total": 80,
  "altura_entrepiso": 2.40,
  "espacios": [
    {
      "nombre": "area_de_ventas",
      "area_m2": 62,
      "actividades": [
        {"capitulo":"CAP01_PRELIMINARES","descripcion":"Localizacion y replanteo","referencia":null,"unidad":"m2","cantidad":62,"suministro":"contratista","origen":"inferida","observacion":null},
        {"capitulo":"CAP01_PRELIMINARES","descripcion":"Desconexion de puntos electricos salidas e iluminacion existentes","referencia":null,"unidad":"gl","cantidad":1,"suministro":"contratista","origen":"inferida","observacion":"supuesto: local comercial con instalaciones anteriores"},
        {"capitulo":"CAP01_PRELIMINARES","descripcion":"Desconexion de puntos hidrosanitarios existentes","referencia":null,"unidad":"gl","cantidad":1,"suministro":"contratista","origen":"inferida","observacion":null},
        {"capitulo":"CAP02_OBRA_CIVIL_ESTRUCTURA","descripcion":"Muro una cara superboard 8mm con pintura acryltex blanca","referencia":null,"unidad":"m2","cantidad":11,"suministro":"contratista","origen":"explicita","observacion":null},
        {"capitulo":"CAP02_OBRA_CIVIL_ESTRUCTURA","descripcion":"Muro doble cara superboard 8mm con pintura acryltex blanca","referencia":null,"unidad":"m2","cantidad":30,"suministro":"contratista","origen":"explicita","observacion":null},
        {"capitulo":"CAP02_OBRA_CIVIL_ESTRUCTURA","descripcion":"Pintura acryltex blanca a 3 manos muros","referencia":"Acryltex blanca","unidad":"m2","cantidad":58,"suministro":"contratista","origen":"inferida","observacion":null},
        {"capitulo":"CAP03_PANETES_ENCHAPES_CIELORASOS","descripcion":"Cieloraso drywall con pintura texturizada Wax Yellow Sapolin","referencia":"Wax Yellow 2015P Sapolin","unidad":"m2","cantidad":62,"suministro":"contratista","origen":"explicita","observacion":null},
        {"capitulo":"CAP03_PANETES_ENCHAPES_CIELORASOS","descripcion":"Tapas de inspeccion cieloraso 60x60 incluye marco aluminio","referencia":null,"unidad":"un","cantidad":2,"suministro":"contratista","origen":"inferida","observacion":"supuesto: tapas de inspeccion para cieloraso drywall"},
        {"capitulo":"CAP03_PANETES_ENCHAPES_CIELORASOS","descripcion":"Piedra sinterizada pared Belvedere Black Mate Gramar","referencia":"Belvedere Black Mate Gramar","unidad":"m2","cantidad":21,"suministro":"contratista","origen":"explicita","observacion":null},
        {"capitulo":"CAP04_PISOS","descripcion":"Afinado de piso","referencia":null,"unidad":"m2","cantidad":62,"suministro":"contratista","origen":"inferida","observacion":null},
        {"capitulo":"CAP04_PISOS","descripcion":"Enchape piso porcelanato Chateau White 120x60 Attmosferas boquilla epoxica","referencia":"Chateau White 120x60 Attmosferas","unidad":"m2","cantidad":62,"suministro":"contratista","origen":"explicita","observacion":null},
        {"capitulo":"CAP04_PISOS","descripcion":"Guardaescoba media cana fundida en sitio con pintura epoxica blanca","referencia":null,"unidad":"ml","cantidad":33,"suministro":"contratista","origen":"inferida","observacion":null},
        {"capitulo":"CAP06_ELECTRICO_DATOS","descripcion":"Salida para lampara lineal","referencia":null,"unidad":"un","cantidad":4,"suministro":"contratista","origen":"explicita","observacion":null},
        {"capitulo":"CAP06_ELECTRICO_DATOS","descripcion":"Salida para tomacorriente doble polo a tierra 15A Leviton","referencia":"Leviton 5262-W","unidad":"un","cantidad":14,"suministro":"contratista","origen":"explicita","observacion":null},
        {"capitulo":"CAP06_ELECTRICO_DATOS","descripcion":"Salida para datos doble CAT 6","referencia":null,"unidad":"un","cantidad":4,"suministro":"contratista","origen":"explicita","observacion":null},
        {"capitulo":"CAP07_HIDROSANITARIO","descripcion":"Punto hidraulico agua fria PVC-P 1/2 Pavco","referencia":"Pavco 1/2\"","unidad":"un","cantidad":6,"suministro":"contratista","origen":"explicita","observacion":null},
        {"capitulo":"CAP07_HIDROSANITARIO","descripcion":"Salida sanitaria PVC 2 pulgadas","referencia":null,"unidad":"un","cantidad":4,"suministro":"contratista","origen":"explicita","observacion":null},
        {"capitulo":"CAP07_HIDROSANITARIO","descripcion":"Instalacion caja lavadora incluye sellado y hermeticidad","referencia":null,"unidad":"un","cantidad":4,"suministro":"contratista","origen":"explicita","observacion":null},
        {"capitulo":"CAP07_HIDROSANITARIO","descripcion":"Instalacion coldrink triple incluye acoples y mangueras","referencia":null,"unidad":"un","cantidad":1,"suministro":"contratista","origen":"explicita","observacion":null},
        {"capitulo":"CAP09_SEGURIDAD_DATOS","descripcion":"Detector de temperatura red de incendios","referencia":null,"unidad":"un","cantidad":2,"suministro":"contratista","origen":"explicita","observacion":null},
        {"capitulo":"CAP09_SEGURIDAD_DATOS","descripcion":"Rociador automatico cobertura extendida","referencia":null,"unidad":"un","cantidad":4,"suministro":"contratista","origen":"explicita","observacion":null},
        {"capitulo":"CAP09_SEGURIDAD_DATOS","descripcion":"Camara CCTV incluye cableado y accesorios","referencia":null,"unidad":"un","cantidad":3,"suministro":"contratista","origen":"explicita","observacion":null}
      ]
    },
    {
      "nombre": "bodega",
      "area_m2": 18,
      "actividades": [
        {"capitulo":"CAP04_PISOS","descripcion":"Afinado de piso bodega","referencia":null,"unidad":"m2","cantidad":18,"suministro":"contratista","origen":"inferida","observacion":null},
        {"capitulo":"CAP10_ADICIONALES","descripcion":"Adecuacion bodega interna","referencia":null,"unidad":"gl","cantidad":1,"suministro":"contratista","origen":"explicita","observacion":null}
      ]
    }
  ],
  "alertas": ["pregunta: los sanitarios los suministra el cliente o el contratista?", "supuesto: altura entrepiso 2.40m, confirmar"]
}

Clasificacion de los capitulos APU:

CAP01_PRELIMINARES
  incluye: todo lo que se desmonta, demolece o retira antes de construir, incluye protección, localización y retiro de escombros.
  Ejemplos: demolición muros drywall, desmonte piso existente, desmonte cieloraso, desmonte tablero eléctrico, desconexión puntos eléctricos e hidrosanitarios,
  regatas en muros y pisos, retiro escombros, material protección.

CAP02_OBRA_CIVIL_ESTRUCTURA
  incluye: construcción de muros nuevos, estructura metálica, mampostería, concreto, cubierta y refuerzos estructurales.
  Ejemplos: muro drywall doble cara con frescasa, muro drywall una cara, muro superboard, muro mampostería, dinteles drywall, estructura metálica
  sobrepiso, cubierta vidrio templado, frescasa muros.

CAP03_PANETES_ENCHAPES_CIELORASOS
  incluye: acabados de muros y techos — pañetes, enchapes, cielorasos, pintura, recubrimientos y elementos decorativos de techo.
  Ejemplos: cieloraso drywall, cieloraso XSound, enchape muro porcelanato, enchape muro cerámica, pintura vinílica, estuco, resanes regatas,
  dilataciones en Z cieloraso, nichos de luz, cortineros drywall, tapas inspección cieloraso 60x60, palillaje melamina techo.

CAP04_PISOS
  incluye: instalación de pisos, alistados, guardaescobas y mediacañas.
  Ejemplos: instalación porcelanato 60x120, instalación cerámica, piso microcemento, alistado de piso, fundida contrapiso, guardaescoba aglomerado, media caña resina,
  guardaescoba metálico, perfil metálico piso.

CAP05_CARPINTERIA_MUEBLES
  incluye: muebles, mesones, carpintería en madera, divisiones en vidrio,fachadas en vidrio/aluminio y acero inoxidable.
  Ejemplos: mueble melamina, mesón piedra sinterizada, mesón acero inoxidable, división vidrio aluminio, puerta corredera vidrio, puerta batiente vidrio,
  fachada vidrio templado, poceta acero inoxidable, carpintería madera.

CAP06_ELECTRICO_DATOS
  incluye: todo lo eléctrico — salidas de iluminación, tomacorrientes, tableros, acometidas, cableado, datos, voz y suministro de luminarias.
  Ejemplos: salida luminaria colgante, salida bala recesada, tomacorriente normal, tomacorriente regulado, tomacorriente GFCI, tablero eléctrico
  trifásico, breakers, acometida, punto de datos CAT6A, patch panel, certificación RETIE, instalación luminaria con suministro.

CAP07_HIDROSANITARIO
  incluye: redes de agua fría, caliente y filtrada, desagües, aparatos sanitarios, griferías, gas y equipos de agua.
  Ejemplos: punto hidráulico agua fría, punto hidráulico agua caliente,
  punto agua filtrada, salida sanitaria 2", salida sanitaria 4", tubería PVC, registro bola, lavamanos, sanitario, grifería, calentador agua, filtro agua,
  punto de gas, coldrink, caja lavadora, trampa grasas.

CAP08_AIRE_ACONDICIONADO
  incluye: ductos, rejillas, difusores, termostatos y equipos de AA.
  Ejemplos: ducto flexible rejilla inyección, ducto flexible rejilla extracción,
  termostato, espiroducto, rejilla lineal, reinstalación rejillas AA.

CAP09_SEGURIDAD_DATOS
  incluye: deteccion de incendios, rociadores, CCTV, control de acceso, sistema de intrusión y red contra incendios.
  Ejemplos: detector de temperatura, rociador automático, estación manual doble acción, tubería EMT red incendios, cámara CCTV, control de acceso,
  sensor magnético, reubicación puntos detección.

CAP10_ADICIONALES
  incluye: bodega, aseo de obra, equipos especiales, elementos que no se encuentran en los capítulos anteriores.
  Ejemplos: adecuación bodega, aseo y entrega de obra, equipo especial,señalización de obra, planos récord.

Reglas de inferencia:
1. Los dos ejemplos anteriores muestran el nivel de detalle y la variedad de actividades esperadas. Cada proyecto es diferente, usar los ejemplos como
referencia de detalle y no como lista fija de actividades.

2. Aplica tu conocimiento y criterio tecnico de interventor en obras constructivas para inferir actividades que se derivan del texto aunque no estén mencionadas explícitamente
3. Marca siempre las actividades inferidas con origen="inferida" y explica el supuesto en el campo observacion.

Reglas generales:
- Para Obra civil o en remodelacion se cumple que remodelación considera desmontes y demoliciones de lo que se va a reemplazar, segun lo que indica el texto
- Para Obra nueva en local comercial se cumple que puede incluir desconexiones de instalaciones anteriores del local
- Usa el tipo de proyecto, el estado de la obra y el contexto del texto para decidir qué actividades adicionales tienen sentido

No apliques reglas fijas y razona caso por caso como lo haria un interventor.

Supuestos:
1. Aplica estos valores cuando el texto no los especifique y agrega la alerta correspondiente en el campo alertas del proyecto.
2. Colocar altura_entrepiso: 2.40m pero se debe generar una alerta de supuesto: altura entrepiso 2.40m, confirmar"
3. Si no hay espacios definidos crear espacio "area_total" con area_total y una alerta de supuesto: proyecto sin espacios, se usó area total"

Preguntas:
Cuando la información no esté en el texto, agrega la pregunta en el campo alertas.

1. Preguntas bloqueantes: se generan cuando se tiene la informacion priomordial para la generacion de la planilla.
  - pregunta_bloqueante: cual es el area total del proyecto en m2?
  - pregunta_bloqueante: el proyecto es Obra Civil/Remodelacion u Obra nueva?
  - pregunta_bloqueante: cual es el tipo de proyecto?

2. Preguntas de alto impacto: se generan cuando son esenciales y pueden generar y una estimacion incorrecta si no se tiene esta informacion
  - pregunta: cuantos m2 tiene (nombre_espacio)?
  - pregunta: cual es la altura del entrepiso?
  - pregunta: cuantos banos tiene el proyecto?
  - pregunta: hay zona de cocina?

3. Detalle: se generan cuando la informacion es necesaria para mejorar la precision
  - pregunta: tiene referencia del material?
  - pregunta: tiene marca del material?

Reglas absolutas:
1. Responde solo con el JSON. Sin texto antes ni despues.
2. Todas las cantidades deben ser numeros mayores a cero, nunca usar null o None en cantidad, si no conoces la cantidad exacta, usa 1 para
   actividades globales (gl) o estima basandote en el area total del proyecto.
3. Nunca inventes referencias de materiales que no esten en el texto.
4. Si el texto menciona una marca o referencia, incluyela en el campo referencia. La descripcion debe contener la actividad tecnica.
5. Usa los valores exactos de los enums para capitulo, unidad, suministro y origen.
6. El nombre del espacio debe estar en minúsculas, sin tildes y con guion bajo
   en lugar de espacios. Ejemplo: "sala_principal", "bano_1", "area_general".
7. Si una actividad puede ir en varios capítulos, elige el más específico.
8. No repitas la misma actividad en el mismo espacio.
9. Desglosa actividades que son TIPOS distintos de trabajo. El campo
   cantidad representa cuántas unidades hay de ese tipo. NO crees una
   actividad por cada unidad individual del mismo tipo de elemento.
   Mal:  12 actividades "instalación tomacorriente normal"
         59 actividades "instalación lámpara spot riel"
   Bien: 1 actividad "instalación tomacorriente normal 15A" cantidad=12
         1 actividad "instalación lámpara tipo spot en riel 18W" cantidad=59

   si desglosa cuando son elementos físicamente diferentes entre si:
   mal:  "Mueble con lavamanos, mesón en sinterizado y grifería con sensor"
   bien: tres actividades separadas porque son elementos distintos:
         - "Mueble inferior en melamina"
         - "Lavamanos de sobreponer con instalación"
         - "Mesón en piedra sinterizada"
10. Usa kg solo para acero estructural u otros elementos que se midan por peso.
11. Las descripciones deben incluir la acción completa más el elemento y su
    especificación técnica. Usa el mismo nivel de detalle que un interventor
    de obra al escribir una planilla APU.
    mal:  "Estructura metálica para riel"
          "Piso porcelanato"
          "Cieloraso"
    bien: "Suministro e instalación de estructura metálica tipo channel ranurado"
          "Instalación enchape piso porcelanato gran formato 60x120"
          "Cieloraso en drywall con frescasa"
12. Jerarquía de actividades a generar:
    primero: extrae todo lo que dice explícitamente el texto con el máximo detalle posible.
    segundo: infiere las actividades que son consecuencia directa y necesaria de lo extraido (si hay piso nuevo entonces alistado previo, si hay muros nuevos
    en remodelación entonces desmonte previo; si hay cieloraso entonces tapas de inspección).
    no generes actividades genéricas del tipo de proyecto si el texto nolas sugiere. Aseo final, señalizacion, material de protección solo si
    el texto o el contexto del proyecto lo justifica claramente.
"""

print("SYSTEM_PROMPT generado correctamente")
print(f"Longitud: {len(SYSTEM_PROMPT)} caracteres")
print(f"Aprox tokens: ~{len(SYSTEM_PROMPT)//4}")

SYSTEM_PROMPT generado correctamente
Longitud: 23004 caracteres
Aprox tokens: ~5751


**Prompt sin Brief:**

In [48]:
SYSTEM_PROMPT_SIN_BRIEF = """
Eres un asistente de construcción. A partir de la descripción de un proyecto
constructivo, genera una planilla de estimación de costos APU en formato JSON.
 
Responde unicamente con un JSON válido con esta estructura:
 
{
  "tipo": "<tipo de proyecto>",
  "estado": "<estado de la obra>",
  "area_total": <número o null>,
  "altura_entrepiso": <número>,
  "espacios": [
    {
      "nombre": "<nombre del espacio>",
      "area_m2": <número o null>,
      "actividades": [
        {
          "capitulo": "<capítulo APU>",
          "descripcion": "<descripción de la actividad>",
          "referencia": "<referencia del material o null>",
          "unidad": "<unidad de medida>",
          "cantidad": <número>,
          "suministro": "<contratista|cliente|por_definir>",
          "origen": "<explicita|inferida>",
          "observacion": null
        }
      ]
    }
  ],
  "alertas": []
}
"""
 

### Herramientas: ###

In [49]:
def validar_pydantic(output_raw, modo):
    try:
        proy = proyecto(**output_raw)
        return True, proy, None
    except Exception as e:
        return False, None, str(e)

In [50]:
def manejar_falla(state, error):
    if state["modo"] == modo_ejecucuion.eval:
        return None
    raw = state.get("output_raw") or {}
    raw.setdefault("alertas", [])
    raw["alertas"].append(
        f"alerta: output incompleto por error de validación — {error}"
    )
    return raw

In [51]:
def generar_preguntas(proyecto):
    alertas = proyecto.alertas if proyecto else []
    bloqueantes  = [a for a in alertas if "pregunta_bloqueante:" in a]
    alto_impacto = [a for a in alertas if a.startswith("pregunta:")]
    supuestos    = proyecto.supuestos if proyecto else []
    return {
        "bloqueantes":       bloqueantes,
        "alto_impacto":      alto_impacto,
        "supuestos":         supuestos,
        "tiene_bloqueantes": len(bloqueantes) > 0,
    }
 

In [52]:
def parsear_json_llm(texto_raw):
    texto = texto_raw.strip()
    if texto.startswith("```"):
        lineas = texto.split("\n")
        texto = "\n".join(
            l for l in lineas if not l.startswith("```")
        ).strip()
    try:
        return json.loads(texto), None
    except json.JSONDecodeError as e:
        return None, str(e)
 

### Langchain: ###

nodos de cada fase de evaluacion:

In [53]:
def nodo_reasoning(state, llm):
    msgs = [SystemMessage(content=SYSTEM_PROMPT), HumanMessage(content=state["texto"])]
    if state["error_pydantic"]:
        msgs.append(HumanMessage(content=(
            f"Tu respuesta anterior tuvo este error:\n{state['error_pydantic']}\nCorrige el JSON y responde nuevamente."
        )))
    t0 = time.time()
    respuesta = llm.invoke(msgs)
    latencia = round((time.time() - t0) * 1000, 2)
    output_raw, error_parse = parsear_json_llm(respuesta.content)
    registrar_trajectory(
        state, "reasoning", "extraer e inferir actividades del texto",
        "ok" if output_raw else "fallo",
        {"intento": state["intentos"] + 1, "error_parse": error_parse, "latencia_ms": latencia, "modelo": state["modelo_llm"]}
    )
    state["output_raw"] = output_raw
    state["latencia_ms"] = latencia
    state["intentos"] += 1
    return state


def nodo_validador(state):
    if state["output_raw"] is None:
        state["schema_ok"] = False
        state["error_pydantic"] = "el LLM no devolvió un JSON válido"
        registrar_trajectory(state, "action_pydantic", "validar schema", "fallo", {"error": state["error_pydantic"]})
        return state
    ok, proy, error = validar_pydantic(state["output_raw"], state["modo"])
    registrar_trajectory(
        state, "action_pydantic", "validar schema",
        "ok" if ok else "fallo",
        {"intento": state["intentos"], "schema_ok": ok, "error": error}
    )
    state["schema_ok"] = ok
    state["error_pydantic"] = error
    state["proyecto"] = proy.model_dump(mode="json") if ok and proy else None
    return state


def nodo_preguntas(state):
    if not state["proyecto"]:
        return state
    proy = proyecto.model_validate(state["proyecto"])
    preguntas = generar_preguntas(proy)
    bloqueado = preguntas["tiene_bloqueantes"] and state["modo"] == modo_ejecucuion.prod
    registrar_trajectory(
        state, "action_preguntas", "generar preguntas pendientes",
        "pregunta" if preguntas["tiene_bloqueantes"] else "ok",
        {"bloqueantes": preguntas["bloqueantes"], "alto_impacto": preguntas["alto_impacto"], "supuestos": preguntas["supuestos"], "bloqueado": bloqueado}
    )
    state["bloqueado"] = bloqueado
    return state


def routing(state):
    if state["schema_ok"]:
        return "preguntas"
    if state["intentos"] < max_intentos:
        return "reasoning"
    state["proyecto"] = manejar_falla(state, state["error_pydantic"])
    return "fin"


Grafos:

In [54]:
def build_grafo(llm):
    reasoning = partial(nodo_reasoning, llm=llm)
    grafo = StateGraph(AgentState)
    grafo.add_node("reasoning", reasoning)
    grafo.add_node("validador", nodo_validador)
    grafo.add_node("preguntas", nodo_preguntas)
    grafo.set_entry_point("reasoning")
    grafo.add_edge("reasoning", "validador")
    grafo.add_conditional_edges(
        "validador",
        routing,
        {
            "reasoning": "reasoning",
            "preguntas": "preguntas",
            "fin":       END,
        }
    )
    grafo.add_edge("preguntas", END)
    return grafo.compile()

Correr el agente:

In [55]:
def run_agente(texto, llm, caso_id=None, modo=modo_ejecucuion.eval):
    agente = build_grafo(llm)
    nombre_modelo = llm.model_name if hasattr(llm, "model_name") else str(llm)
    state = init_state(texto=texto, modo=modo, caso_id=caso_id, modelo_llm=nombre_modelo)
    return agente.invoke(state)

### Normalizacion del Ground T: ###

In [56]:
col_item = 0
col_desc = 1
col_unid = 2
col_cant = 3


def es_capitulo(item):
    if item is None:
        return False
    try:
        val = float(str(item).strip())
        return val == int(val) and val > 0
    except (ValueError, TypeError):
        return False


def normalizar_item(item):
    if item is None:
        return None
    return str(item).strip().rstrip(".")


def parsear_planilla(ruta_excel, caso_id, nombre):
    wb = openpyxl.load_workbook(ruta_excel, data_only=True)
    hoja = wb.active
    capitulos = {}
    cap_actual = "SIN_CAPITULO"

    for fila in hoja.iter_rows(values_only=True):
        item = fila[col_item] if len(fila) > col_item else None
        desc = fila[col_desc] if len(fila) > col_desc else None
        unid = fila[col_unid] if len(fila) > col_unid else None
        cant = fila[col_cant] if len(fila) > col_cant else None
        if item is None and desc is None:
            continue

        if item is not None:
            try:
                float(str(item).strip())
            except (ValueError, TypeError):
                continue

        if es_capitulo(item) and desc:
            cap_nombre = str(desc).strip()
            cap_actual = norm_capitulos_fn(cap_nombre)
            if cap_actual not in capitulos:
                capitulos[cap_actual] = {"nombre_original": cap_nombre, "actividades": []}
            continue

        if desc and str(desc).strip():
            try:
                cantidad = float(str(cant).replace(",", ".")) if cant else 0.0
            except (ValueError, TypeError):
                cantidad = 0.0

            unidad = norm_texto(str(unid).strip()) if unid else "un"
            unidad = norm_unidad.get(unidad, unidad)

            actividad = {
                "item": normalizar_item(item),
                "descripcion": str(desc).strip(),
                "unidad": unidad,
                "cantidad": round(cantidad, 2),
            }

            if cap_actual not in capitulos:
                capitulos[cap_actual] = {"nombre_original": cap_actual, "actividades": []}
            capitulos[cap_actual]["actividades"].append(actividad)

    return {"caso_id": caso_id, "nombre": nombre, "capitulos": capitulos}


def resumen_gt(gt):
    print(f"\n{gt['nombre']} — {gt['caso_id']}")
    total = 0
    for cap, data in gt["capitulos"].items():
        n = len(data["actividades"])
        total += n
        print(f"  {cap:<45} {n} actividades")
    print(f"  {'TOTAL':<45} {total}")

### Evaluación: ###

Se van a realizar todas las funciones correspondientes a cada una de las fases de evaluación del modelo:

In [57]:
modelo_embed = SentenceTransformer('paraphrase-multilingual-mpnet-base-v2')

umbral_match = 0.70

pesos = {
    "schema_ok": 0.10,
    "recall_global": 0.25,
    "precision_global": 0.10,
    "f1_capitulos": 0.05,
    "mape_cantidades": 0.10,
    "trajectory_match": 0.25,
    "judge_avg": 0.15,
}

@dataclass
class EvalScores:
    caso_id: str
    modelo_llm: str
    config: str
    schema_ok: bool = False
    n_intentos: int = 0
    n_fallos_schema: int = 0
    estado_ok: bool = False
    tipo_ok: bool = False
    area_mape: float = 0.0
    tiene_metadata: bool = False
    recall_global: float = 0.0
    precision_global: float = 0.0
    f1_global: float = 0.0
    f1_capitulos: float = 0.0
    mape_cantidades: float = 0.0
    n_pares_mape: int = 0
    unidad_accuracy: float = 0.0
    n_matched: int = 0
    n_pred: int = 0
    n_gt: int = 0
    cobertura: float = 0.0
    trajectory_score: float = 0.0
    chapter_recall: float = 0.0
    trajectory_match: float = 0.0
    n_preguntas: int = 0
    n_bloqueantes: int = 0
    n_supuestos: int = 0
    tuvo_preguntas: bool = False
    correctness: float = 0.0
    faithfulness: float = 0.0
    efficiency: float = 0.0
    judge_avg: float = 0.0
    score_global: float = 0.0
    latencia_ms: float = 0.0
    errores: List[str] = field(default_factory=list)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4929.64it/s]


***Funciones para calcular:** 
 - matching entre el GT y el resultado del agente en actividades
 - F1 de capitulos
 - MAPE para areas totales
 - acurracy de unidade de medida

In [58]:
def calcular_schema(state):
    return state.get("schema_ok", False)


def matching_actividades(preds, gts, umbral=umbral_match):
    if not preds or not gts:
        return []

    emb_pred = modelo_embed.encode(preds, convert_to_tensor=True)
    emb_gt = modelo_embed.encode(gts, convert_to_tensor=True)
    matriz = util.cos_sim(emb_gt, emb_pred).numpy()
    matches = []
    pred_usados = set()
    for i in range(len(gts)):
        fila = [(matriz[i][j], j) for j in range(len(preds)) if j not in pred_usados]
        if not fila:
            continue
        score, j = max(fila)
        if score >= umbral:
            matches.append((j, i, float(score)))
            pred_usados.add(j)
    return matches


def calcular_retrieval(preds_planas, gt_capitulos):
    gt_all = []
    for cap_data in gt_capitulos.values():
        for act in cap_data["actividades"]:
            gt_all.append(act)
    if not gt_all or not preds_planas:
        return 0.0, 0.0, 0.0, []

    descs_pred = [f"{a['descripcion']} {a.get('referencia') or ''}".strip() for a in preds_planas]
    descs_gt = [a["descripcion"] for a in gt_all]
    matches = matching_actividades(descs_pred, descs_gt)
    n_matches = len(matches)
    recall = round(n_matches / len(gt_all), 4)
    precision = round(n_matches / len(preds_planas), 4) if preds_planas else 0.0
    f1 = round(2 * precision * recall / (precision + recall), 4) if (precision + recall) > 0 else 0.0
    return recall, precision, f1, matches


def calcular_f1_capitulos(preds_planas, gt_capitulos):
    caps_gt = set(c for c in gt_capitulos if c != "SIN_CAPITULO" and gt_capitulos[c]["actividades"])
    caps_pred = set(a["capitulo"] for a in preds_planas)
    if not caps_gt:
        return 0.0, 0.0

    tp = len(caps_gt & caps_pred)
    fp = len(caps_pred - caps_gt)
    fn = len(caps_gt - caps_pred)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    chapter_recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (2 * precision * chapter_recall / (precision + chapter_recall)
          if (precision + chapter_recall) > 0 else 0.0)
    return round(f1, 4), round(chapter_recall, 4)


def calcular_mape(preds_planas, gt_capitulos, matches):
    if not matches:
        return 0.0, 0

    gt_all = [act for cap in gt_capitulos.values() for act in cap["actividades"]]
    errores = []
    pares_validos = 0

    for idx_pred, idx_gt, _ in matches:
        if idx_pred >= len(preds_planas) or idx_gt >= len(gt_all):
            continue
        cant_pred = preds_planas[idx_pred]["cantidad"]
        cant_gt = gt_all[idx_gt]["cantidad"]
        unit_pred = str(preds_planas[idx_pred].get("unidad", "")).strip()
        unit_gt = str(gt_all[idx_gt].get("unidad", "")).strip()

        if cant_gt and cant_gt > 0 and unit_pred == unit_gt:
            errores.append(abs(cant_pred - cant_gt) / cant_gt)
            pares_validos += 1

    mape = round(float(np.mean(errores)), 4) if errores else 0.0
    return mape, pares_validos


def calcular_unidad_accuracy(preds_planas, gt_capitulos, matches):
    if not matches:
        return 0.0

    gt_all = [act for cap in gt_capitulos.values() for act in cap["actividades"]]
    correctas = 0
    total = 0

    for idx_pred, idx_gt, _ in matches:
        if idx_pred >= len(preds_planas) or idx_gt >= len(gt_all):
            continue
        unit_pred = str(preds_planas[idx_pred].get("unidad", "")).strip().lower()
        unit_gt = str(gt_all[idx_gt].get("unidad", "")).strip().lower()
        if unit_pred and unit_gt:
            total += 1
            if unit_pred == unit_gt:
                correctas += 1
    return round(correctas / total, 4) if total > 0 else 0.0

Cargar el excel que cuenta con toda la información general de cada uno de los proyectos, este excel cuenta con el nombre del proyecto, tipo de proyecto, estado del proyecto y area total:

In [59]:
def cargar_metadata_proyectos(ruta_excel):
    import openpyxl
    def norm_nombre(s):
        return str(s).strip().lower().replace(" ", "_")

    def norm_estado(s):
        s = str(s).strip().lower()
        if "nueva" in s or "gris" in s:
            return "obra nueva"
        return "remodelacion o obra civil"

    def norm_tipo(s):
        s = str(s).strip().lower()
        if "oficina" in s:
            return "oficina"
        if "restaurante" in s:
            return "restaurante"
        if any(x in s for x in ["consultorio", "clinica", "medicina", "celula", "dolor"]):
            return "consultorio"
        if "tienda" in s:
            return "tienda"
        if "apartamento" in s or "apto" in s:
            return "apartamento"
        return None

    try:
        wb = openpyxl.load_workbook(ruta_excel, data_only=True)
        ws = wb.active
        metadata = {}
        for i, row in enumerate(ws.iter_rows(values_only=True)):
            if i == 0:
                continue
            nombre, estado, tipo, area, *_ = row
            if nombre is None:
                continue
            metadata[norm_nombre(nombre)] = {
                "estado_gt": norm_estado(str(estado)) if estado else None,
                "tipo_gt": norm_tipo(str(tipo)) if tipo else None,
                "area_total_gt": float(area) if area else None,
            }
        print(f"[metadata] {len(metadata)} proyectos cargados")
        return metadata
    except Exception as e:
        print(f"[metadata] error cargando Excel: {e}")
        return {}

def agregar_metadata_a_gt(gt_map, metadata_proyectos):
    def norm(s):
        return str(s).strip().lower().replace(" ", "_")

    enriquecidos = 0
    for caso_id, gt in gt_map.items():
        nombre_norm = norm(gt.get("nombre", ""))
        if nombre_norm in metadata_proyectos:
            gt["metadata"] = metadata_proyectos[nombre_norm]
            enriquecidos += 1
        else:
            gt["metadata"] = None
    print(f"[metadata] {enriquecidos}/{len(gt_map)} casos enriquecidos")
    return gt_map

**Funciones para evaluar:**
- estado, tipo de proyecto, area total
- trayectory
- trayectory match

In [60]:
def calcular_metricas_proyecto(proy, metadata):
    if not metadata or not proy:
        return False, False, 0.0, False

    estado_ok = False
    tipo_ok = False
    area_mape = 0.0

    if metadata.get("estado_gt"):
        estado_ok = (proy.estado.value == metadata["estado_gt"])

    if metadata.get("tipo_gt"):
        tipo_ok = (proy.tipo.value == metadata["tipo_gt"])

    if metadata.get("area_total_gt") and proy.area_total and proy.area_total > 0:
        area_gt = metadata["area_total_gt"]
        area_pred = proy.area_total
        area_mape = round(abs(area_pred - area_gt) / area_gt, 4)

    return estado_ok, tipo_ok, area_mape, True


def calcular_trajectory(trajectory_agente):
    n_intentos = 0
    n_fallos = 0
    n_preguntas = 0
    n_bloqueantes = 0
    n_supuestos = 0
    tuvo_preguntas = False

    for paso in trajectory_agente:
        comp = paso.get("componente", "")
        res = paso.get("resultado", "")
        det = paso.get("detalle", {})

        if comp == "reasoning":
            n_intentos += 1
        if comp == "action_pydantic" and res == "fallo":
            n_fallos += 1
        if comp == "action_preguntas":
            n_bloqueantes += len(det.get("bloqueantes", []))
            n_preguntas += len(det.get("alto_impacto", []))
            n_supuestos += len(det.get("supuestos", []))
            if res == "pregunta":
                tuvo_preguntas = True

    score = max(0.0, round(1.0 - n_fallos * 0.2, 2)) if n_intentos > 0 else 0.0

    return {
        "n_intentos": n_intentos,
        "n_fallos": n_fallos,
        "n_preguntas": n_preguntas,
        "n_bloqueantes": n_bloqueantes,
        "n_supuestos": n_supuestos,
        "tuvo_preguntas": tuvo_preguntas,
        "score": score,
    }


def calcular_trajectory_match(trajectory_efficiency, chapter_recall):
    return round(0.5 * trajectory_efficiency + 0.5 * chapter_recall, 4)



## Funciones para generar las tablas de resultados: ##
- Evaluacion calidad de la salida
- Evaluacion metricas generales del proyecto
- Evaluacion Ciclo React
- Evaluacion LLM-AS-JUDGE
- Evaluacion de experimentos

Prompot del juez evaluador:

In [61]:
prompt_juez = """
Eres un evaluador experto en estimación de costos de construcción en Colombia.

Evalúa el output de un agente que genera planillas APU a partir de descripciones
de proyectos constructivos. Se te proporciona:
  1. El texto original del proyecto
  2. La planilla generada por el agente
  3. Un resumen de la planilla real del interventor (ground truth), con las
     primeras actividades de cada capítulo como referencia

Evalúa en escala de 1 a 5:

CORRECTNESS: comparando con el ground truth, ¿el agente cubrió las actividades
correctas del proyecto?
  5 = captura las actividades principales del GT sin errores importantes
  3 = falta o sobra algunas actividades relevantes del GT
  1 = errores graves o faltantes críticos respecto al GT

FAITHFULNESS: ¿todo lo que generó tiene soporte en el texto o es inferencia válida?
  5 = sin alucinaciones, todo viene del texto o del conocimiento del dominio
  3 = algunas actividades sin soporte claro en el texto
  1 = muchas actividades inventadas sin soporte

EFFICIENCY: ¿las preguntas generadas y supuestos aplicados son razonables para
este tipo de proyecto?
  5 = preguntas necesarias, supuestos correctos para el tipo de proyecto
  3 = algunos supuestos cuestionables o preguntas innecesarias
  1 = supuestos incorrectos o preguntas que no aportan valor

Responde unicamente con este JSON:
{
  "correctness":  <1-5>,
  "faithfulness": <1-5>,
  "efficiency":   <1-5>,
  "justificacion": "<una línea>"
}
"""

In [62]:
def formatear_gt_para_juez(gt_capitulos):
    lineas = []
    for cap, data in gt_capitulos.items():
        acts = data["actividades"]
        if not acts:
            continue
        n = len(acts)
        muestra = [a["descripcion"][:55] for a in acts[:2]]
        resto = f" (+{n-2} más)" if n > 2 else ""
        lineas.append(f"{cap} ({n}): {', '.join(muestra)}{resto}")
    return "\n".join(lineas)


def llm_as_judge(texto, output_agente, gt_capitulos, llm_juez):
    gt_texto = formatear_gt_para_juez(gt_capitulos)
    agente_texto = "\n".join(
        f"{a['capitulo']}: {a['descripcion']} | {a['cantidad']} {a['unidad']}"
        for a in output_agente
    ) if output_agente else "Sin output"

    msgs = [
        SystemMessage(content=prompt_juez),
        HumanMessage(content=f"TEXTO:\n{texto}\n\nPLANILLA AGENTE:\n{agente_texto}\n\nGROUND TRUTH:\n{gt_texto}")
    ]

    try:
        respuesta = llm_juez.invoke(msgs)
        resultado, _ = parsear_json_llm(respuesta.content)
        if resultado:
            c = float(resultado.get("correctness", 0))
            f = float(resultado.get("faithfulness", 0))
            e = float(resultado.get("efficiency", 0))
            return c, f, e
    except Exception:
        pass
    return 0.0, 0.0, 0.0

def calcular_score_global(scores):
    if not scores.schema_ok:
        return 0.0

    mape_norm = max(0.0, 1.0 - scores.mape_cantidades)
    judge_norm = scores.judge_avg / 5.0

    return round(
        pesos["schema_ok"] * 1.0 +
        pesos["recall_global"] * scores.recall_global +
        pesos["precision_global"] * scores.precision_global +
        pesos["f1_capitulos"] * scores.f1_capitulos +
        pesos["mape_cantidades"] * mape_norm +
        pesos["trajectory_match"] * scores.trajectory_match +
        pesos["judge_avg"] * judge_norm,
        4
    )

def actividades_planas_sin_pydantic(output_raw):
    if not output_raw:
        return []
    return [
        {
            "espacio": esp.get("nombre", ""),
            "capitulo": act.get("capitulo", ""),
            "descripcion": act.get("descripcion", ""),
            "referencia": act.get("referencia"),
            "unidad": act.get("unidad", ""),
            "cantidad": act.get("cantidad", 0),
            "suministro": act.get("suministro", ""),
            "origen": act.get("origen", ""),
        }
        for esp in output_raw.get("espacios", [])
        for act in esp.get("actividades", [])
    ]

In [63]:
def evaluar_caso(texto, state, gt, llm_juez, config="con_brief", usar_pydantic=True):
    scores = EvalScores(
        caso_id=gt["caso_id"],
        modelo_llm=state.get("modelo_llm", ""),
        config=config,
        latencia_ms=state.get("latencia_ms", 0.0) or 0.0,
    )

    scores.schema_ok = calcular_schema(state)
    scores.n_gt = sum(len(v["actividades"]) for v in gt["capitulos"].values())

    traj = calcular_trajectory(state.get("trajectory", []))
    scores.n_intentos = traj["n_intentos"]
    scores.n_fallos_schema = traj["n_fallos"]
    scores.n_preguntas = traj["n_preguntas"]
    scores.n_bloqueantes = traj["n_bloqueantes"]
    scores.n_supuestos = traj["n_supuestos"]
    scores.tuvo_preguntas = traj["tuvo_preguntas"]
    scores.trajectory_score = traj["score"]

    if usar_pydantic:
        if not scores.schema_ok:
            scores.errores.append("falla de schema")
            scores.score_global = calcular_score_global(scores)
            return scores
        proy = proyecto(**state["proyecto"])
        preds_planas = proy.actividades_planas
    else:
        preds_planas = actividades_planas_sin_pydantic(state.get("output_raw"))
    scores.n_pred = len(preds_planas)
    scores.cobertura = round(scores.n_pred / scores.n_gt, 4) if scores.n_gt > 0 else 0.0

    scores.recall_global, scores.precision_global, scores.f1_global, matches = calcular_retrieval(preds_planas, gt["capitulos"])
    scores.f1_capitulos, scores.chapter_recall = calcular_f1_capitulos(preds_planas, gt["capitulos"])
    scores.mape_cantidades, scores.n_pares_mape = calcular_mape(preds_planas, gt["capitulos"], matches)
    scores.n_matched = len(matches)
    scores.unidad_accuracy = calcular_unidad_accuracy(preds_planas, gt["capitulos"], matches)
    scores.trajectory_match = calcular_trajectory_match(scores.trajectory_score, scores.chapter_recall)

    metadata = gt.get("metadata")
    if metadata and usar_pydantic and scores.schema_ok:
        scores.estado_ok, scores.tipo_ok, scores.area_mape, scores.tiene_metadata = calcular_metricas_proyecto(proy, metadata)

    scores.correctness, scores.faithfulness, scores.efficiency = llm_as_judge(texto, preds_planas, gt["capitulos"], llm_juez)
    scores.judge_avg = round((scores.correctness + scores.faithfulness + scores.efficiency) / 3, 4)
    scores.score_global = calcular_score_global(scores)
    return scores

In [64]:
def evaluar_corpus(corpus, llm, llm_juez, gt_map, config="con_brief", usar_pydantic=True):
    resultados = []
    for caso in corpus:
        if caso["caso_id"] not in gt_map:
            continue
        print(f"evaluando {caso['caso_id']} ({caso['nombre']})...", end=" ")
        state = run_agente(texto=caso["texto"], llm=llm, caso_id=caso["caso_id"])
        scores = evaluar_caso(
            texto=caso["texto"], state=state, gt=gt_map[caso["caso_id"]],
            llm_juez=llm_juez, config=config, usar_pydantic=usar_pydantic,
        )
        resultados.append(scores)
        print(
            f"schema={scores.schema_ok} | "
            f"recall={scores.recall_global:.3f} prec={scores.precision_global:.3f} | "
            f"cap_r={scores.chapter_recall:.3f} traj={scores.trajectory_match:.3f} | "
            f"score={scores.score_global:.3f}"
        )
    return resultados


In [65]:
def tabla_calidad_salida(resultados):
    print("TABLA 5.1:  CALIDAD DE LA SALIDA")
    for r in resultados:
        print(r.caso_id, "recall:", r.recall_global, "prec:", r.precision_global,
              "f1:", r.f1_global, "f1cap:", r.f1_capitulos,
              "mape:", r.mape_cantidades, "u_acc:", r.unidad_accuracy, "cob:", r.cobertura)

    n_mape_tot = sum(r.n_pares_mape for r in resultados)
    casos_mape = [r for r in resultados if r.n_pares_mape > 0]
    mape_prom = np.mean([r.mape_cantidades for r in casos_mape]) if casos_mape else 0.0
    print("PROMEDIO",
          "recall:", round(np.mean([r.recall_global for r in resultados]), 3),
          "prec:", round(np.mean([r.precision_global for r in resultados]), 3),
          "f1:", round(np.mean([r.f1_global for r in resultados]), 3),
          "f1cap:", round(np.mean([r.f1_capitulos for r in resultados]), 3),
          "mape:", round(mape_prom, 3), f"({n_mape_tot}p)",
          "u_acc:", round(np.mean([r.unidad_accuracy for r in resultados]), 3),
          "cob:", round(np.mean([r.cobertura for r in resultados]), 3))

    con_meta = [r for r in resultados if r.tiene_metadata]
    if con_meta:
        print(f"TABLA 5.1b: METRICAS DE PROYECTO (n={len(con_meta)})")
        for r in con_meta:
            print(r.caso_id,
                  "estado_ok:", r.estado_ok,
                  "tipo_ok:", r.tipo_ok,
                  "area_mape:", r.area_mape)
        estado_acc = sum(1 for r in con_meta if r.estado_ok) / len(con_meta)
        tipo_acc = sum(1 for r in con_meta if r.tipo_ok) / len(con_meta)
        print("ACCURACY estado:", round(estado_acc, 3),
              "tipo:", round(tipo_acc, 3),
              "area_mape:", round(np.mean([r.area_mape for r in con_meta]), 3))
      
def tabla_ciclo_react(resultados):
    print("TABLA 5.2: CICLO REACT Y TRAJECTORY MATCH")
    for r in resultados:
        print(r.caso_id,
              "schema:", r.schema_ok,
              "intentos:", r.n_intentos,
              "fallos:", r.n_fallos_schema,
              "preguntas:", r.n_preguntas,
              "bloq:", r.n_bloqueantes,
              "sup:", r.n_supuestos,
              "eff:", r.trajectory_score,
              "cap_r:", r.chapter_recall,
              "traj_m:", r.trajectory_match)
    schema_pct = sum(1 for r in resultados if r.schema_ok) / len(resultados)
    print("PROMEDIO schema:", round(schema_pct, 3),
          "intentos:", round(np.mean([r.n_intentos for r in resultados]), 2),
          "fallos:", round(np.mean([r.n_fallos_schema for r in resultados]), 2),
          "effic:", round(np.mean([r.trajectory_score for r in resultados]), 2),
          "cap_recall:", round(np.mean([r.chapter_recall for r in resultados]), 2),
          "traj_m:", round(np.mean([r.trajectory_match for r in resultados]), 3))
    
def tabla_llm_judge(resultados):
    print("TABLA 5.3 - LLM-AS-JUDGE")
    for r in resultados:
        print(r.caso_id,
              "correctness:", r.correctness,
              "faithfulness:", r.faithfulness,
              "efficiency:", r.efficiency,
              "avg:", r.judge_avg,
              "score:", r.score_global)
    print("PROMEDIO",
          "correctness:", round(np.mean([r.correctness for r in resultados]), 2),
          "faithfulness:", round(np.mean([r.faithfulness for r in resultados]), 2),
          "efficiency:", round(np.mean([r.efficiency for r in resultados]), 2),
          "avg:", round(np.mean([r.judge_avg for r in resultados]), 2),
          "score:", round(np.mean([r.score_global for r in resultados]), 3))
    
def tabla_ablation(resultados_lista, nombres):
    print("TABLA - VALIDACION DE EXPERIMENTOS")
    for resultados, nombre in zip(resultados_lista, nombres):
        schema_pct = sum(1 for r in resultados if r.schema_ok) / len(resultados)
        print(nombre,
              "schema:", round(schema_pct, 3),
              "recall:", round(np.mean([r.recall_global for r in resultados]), 3),
              "prec:", round(np.mean([r.precision_global for r in resultados]), 3),
              "f1cap:", round(np.mean([r.f1_capitulos for r in resultados]), 3),
              "cap_r:", round(np.mean([r.chapter_recall for r in resultados]), 2),
              "traj_m:", round(np.mean([r.trajectory_match for r in resultados]), 3),
              "u_acc:", round(np.mean([r.unidad_accuracy for r in resultados]), 3),
              "faith:", round(np.mean([r.faithfulness for r in resultados]), 2),
              "score:", round(np.mean([r.score_global for r in resultados]), 3))


In [66]:
def reporte_completo(resultados):
    tabla_calidad_salida(resultados)
    tabla_ciclo_react(resultados)
    tabla_llm_judge(resultados)

### Ejecución del Modelo : ###

### Prueba 1: ###

In [67]:
# # ── INPUT ─────────────────────────────────────────────────────

# texto_bien_plantado = """
# Es la adecuación de un espacio completamente nuevo,  recibido en obra gris de 66m2,  Partió de la construcción de un sobre piso en estructura metálica de 18cm de altura  con un piso en lamina de fibrocemento de 20mm,  y un piso cerámico de gran formato 60x120 de marca Ceramica Italia Cernatto Jaspe.  Los muros serán enchapados en un brick  de prosein  10x30 que será instalado en módulos de 6 dilatados con una dilatación metálica marca Atrim de 1cm.  El mobiliaro se compondrá por 6 muebles en melamina marca duratex,  el mueble DM1  es un muelbe inferior de 5.80m  que tendrá 3 niveles,  el primer nivel  tendrá 2 lavamanos de sobreponer con grifería tipo lavaplatos con sensor,  el segundo nivel tiene un lavaplatos doble en acero inoxidable marca socoda  y el tercer nivel tendrá el espacio para la instalación de una lavavajillas y una lavadora secadora de carga frontal, la profundidad del mueble es de 75cm  y  el mesón es en una piedra sinterizada marca granitos y mármoles  referencia torano oro,  el lavaplatos contará con una llave con monocontrol para agua fría y caliente y una llave independiente para el sistema de agua filtrada,  se contará con cuatro tomacorrientes sobre el mesón,  mas las tomas necesarias para la conexión del lavavajilals y lavadora; el mueble dm2  es una isla de 4.8m x 1.5m  que esta diseñada para tener dos estufas de 4 puestos,  en el centro un lavaplatos redondo de acero inoxidable con una llave monocontrol para agua fría y caliente  y una llave independiente para el sistema de agua filtrada,  se debe contar con dos tomacorrientes para los chisperos de las estufas  y  en una de las esquinas del mueble se instalará un horno empotrable bifásico;  del mueble dm2 serán dos unidades;  EL mueble DM3 tiene 7.5x 52cm consta de un mueble interior y uno exterior,  el interior se construirá en lamina de cold-rolled calibre 18 y tendrá puertas con chapas de seguridad,  este mueble será forrado con melamina RH de 18mm, y en la parte superior tendrá un mesón de 3mx50cm en sinterizado referencia Torano Oro,  la parte exterior del mueble se compondrá de una barra  que tiene una parte curva, en la misma melamina.  En uno de los costados se instalará un panel en melamina que cubrirá el soporte del televisor y un espacio de 90cm para la instalación de un filtro de agua, eL mueble tendrá integrado una superficie que hará las veces de escritorio de 1.20m de largo para el computador de atención al cliente,  se tendrán 5 tomas en el muebe hacia la parte interior y 4 tomas en la parte exterior,  el escritorio contará con dos tomas y una adicional para la instalación de un filtro de agua y otra para el televisor; el mueble DM4 constará de una alacena que alojará una pantalla de 32" de cara al público, unas alacenas que estarán cerradas con vidrio laminado,  el espacio para el nevecon el cual contará con una salida para agua filtrada, con unas repisas superiores iluminadas con cinta LED  y  un mueble que permitirá empotrar dos hornos, con un mesón en melamina,  este mueble cubrirá las columnas metálicas, se requiere tomacorriente para la conexión del monitor, el nevecon y dos freidoras que estarán en el mesón,  adicioanlmente la conexión para los dos hornos bifásicos;  EL mueble DM5  será un mueble en melamina con mesón en acero inoxidable  el cual tendrá integrada una poceta doble con llave monocontrol para agua fría y caliente.  El mueble DM6 es una columna semicircular que tendrá como función cubrir el tablero eléctrico y la tubería que va sobrepuesta,  contará con 3 repisas que tendrán tomacorrientes que servirán de estación de carga de celulares,  en el mueble también estará el interruptor triple para la iluminación del local.
# El proyecto contempla la instalación de un calentador de agua de 13kw  y un filtro de agua everpure de 3 etapas,  así como una poceta lavatraperos en acero inoxidable de 50x50cm.
# La iluminación constará de 20 salidas de iluminación distribuidias en 5 lineas x 4 luminarias en el salón principal,  una salida en el espacio del malacate y poceta lavatraperos,  una spot de iluminación para la zona de lavado en el mueble dm5 y 4 lamparas para control de insectos.
# """

# # ── CORRER EL AGENTE ──────────────────────────────────────────

# resultado = run_agente(
#     texto   = texto_bien_plantado,
#     llm     = llm_gpt,
#     caso_id = "caso_05"
# )

# if resultado["schema_ok"]:
#     proy = proyecto(**resultado["proyecto"])
#     print(proy.planilla_texto())
#     print(f"\nactividades totales: {len(proy.actividades_planas)}")
#     print(f"preguntas pendientes: {proy.preguntas_pendientes}")
# else:
#     print("falla de schema:", resultado["error_pydantic"])

# # ── VER TRAYECTORIA ───────────────────────────────────────────

# print("\nTRAYECTORIA:")
# for paso in resultado["trajectory"]:
#     print(f"  [{paso['componente']}] {paso['decision']} → {paso['resultado']}")

# # ── PARSEAR GROUND TRUTH ──────────────────────────────────────

# gt = parsear_planilla(
#     ruta_excel = r"C:\Users\Dell\Downloads\Tesis\bien_plantado.xlsx",
#     caso_id    = "caso_05",
#     nombre     = "Bien Plantado"
# )
# resumen_gt(gt)

# for act in gt["capitulos"].get("NO_MAPEADO", {}).get("actividades", []):
#     print(act["item"], act["descripcion"][:50])

# # ── EVALUAR ───────────────────────────────────────────────────

# scores = evaluar_caso(
#     texto         = texto_bien_plantado,
#     state         = resultado,
#     gt            = gt,
#     anotaciones   = anotaciones,
#     trajectory_gt = [],
#     llm_juez      = llm_juez,
#     config        = "con_brief"
# )

# print(f"\nRESULTADOS CASO 05 — Bien Plantado")
# print(f"schema_ok:          {scores.schema_ok}")
# print(f"recall_explicitas:  {scores.recall_explicitas}")
# print(f"recall_inferidas:   {scores.recall_inferidas}")
# print(f"f1_capitulos:       {scores.f1_capitulos}")
# print(f"mape_cantidades:    {scores.mape_cantidades}")
# print(f"trajectory_match:   {scores.trajectory_match}")
# print(f"correctness:        {scores.correctness}")
# print(f"faithfulness:       {scores.faithfulness}")
# print(f"efficiency:         {scores.efficiency}")
# print(f"score_global:       {scores.score_global}")

## Procesamiento del Input: ##
Descripciones textuales de cada uno de los proyectos

In [68]:
from docx import Document


def leer_corpus_word(ruta_docx):
    doc  = Document(ruta_docx)
    textos = {}
    caso_actual = None
    parrafos = []

    for p in doc.paragraphs:
        texto = p.text.strip()
        if not texto:
            continue

        texto_lower = texto.lower().replace(" ", "")
        if texto_lower.startswith("proyecto:") or texto_lower.startswith("´proyecto:"):
            if caso_actual and parrafos:
                textos[caso_actual] = " ".join(parrafos).strip()
            nombre = texto.split(":", 1)[1].strip()
            nombre = nombre.replace("´", "").strip()
            caso_actual = nombre
            parrafos = []
        else:
            if caso_actual:
                parrafos.append(texto)
    if caso_actual and parrafos:
        textos[caso_actual] = " ".join(parrafos).strip()
    return textos


RUTA_BASE = r"C:\Users\Dell\Downloads\Tesis"
RUTA_PLANILLAS = r"C:\Users\Dell\Downloads\Tesis\data"           
RUTA_META = r"C:\Users\Dell\Downloads\Tesis\proyectos_generales.xlsx"  
ruta_word = f"{RUTA_BASE}\\ProyectosPring.docx"

CASOS_EXCLUIDOS = {"Relive", "juan_valdez_and"}

textos_word = leer_corpus_word(ruta_word)

corpus = []
gt_map = {}

for i, (nombre, texto) in enumerate(textos_word.items(), start=1):
    caso_id = f"caso_{str(i).zfill(2)}"
    ruta_xlsx = f"{RUTA_PLANILLAS}\\{nombre}.xlsx"  

    corpus.append({
        "caso_id": caso_id,
        "nombre": nombre,
        "texto": texto,
    })

    if os.path.exists(ruta_xlsx):
        gt_map[caso_id] = parsear_planilla(ruta_xlsx, caso_id, nombre)
    else:
        print(f"advertencia: sin planilla para '{nombre}'")

corpus        = sorted(corpus, key=lambda x: x["caso_id"])
corpus_con_gt = [c for c in corpus if c["caso_id"] in gt_map]
corpus_con_gt = [c for c in corpus_con_gt
                 if c["nombre"] not in CASOS_EXCLUIDOS]

metadata_proyectos = cargar_metadata_proyectos(RUTA_META)
gt_map = agregar_metadata_a_gt(gt_map, metadata_proyectos)

print(f"corpus total: {len(corpus)} casos")
print(f"corpus evaluación: {len(corpus_con_gt)} casos")
print(f"excluidos: {CASOS_EXCLUIDOS}")

advertencia: sin planilla para 'consultorio'
[metadata] 26 proyectos cargados
[metadata] 25/25 casos enriquecidos
corpus total: 26 casos
corpus evaluación: 23 casos
excluidos: {'juan_valdez_and', 'Relive'}


### Validaciones apartir del Brief y Pydantic: ###

In [69]:
def run_agente_exp(texto, llm, caso_id=None, modo=modo_ejecucuion.eval, prompt=None):
    prompt_usar = prompt if prompt else SYSTEM_PROMPT
    agente = build_grafo_exp(llm, prompt_usar)
    nombre_modelo = llm.model_name if hasattr(llm, "model_name") else str(llm)
    state = init_state(texto=texto, modo=modo, caso_id=caso_id, modelo_llm=nombre_modelo)
    return agente.invoke(state)


def build_grafo_exp(llm, prompt):
    grafo = StateGraph(AgentState)
    grafo.add_node("reasoning", lambda s: nodo_reasoning_exp(s, llm, prompt))
    grafo.add_node("validador", nodo_validador)
    grafo.add_node("preguntas", nodo_preguntas)
    grafo.set_entry_point("reasoning")
    grafo.add_edge("reasoning", "validador")
    grafo.add_conditional_edges("validador", routing,
        {"reasoning": "reasoning", "preguntas": "preguntas", "fin": END})
    grafo.add_edge("preguntas", END)
    return grafo.compile()


def nodo_reasoning_exp(state, llm, prompt):
    msgs = [SystemMessage(content=prompt), HumanMessage(content=state["texto"])]
    if state["error_pydantic"]:
        msgs.append(HumanMessage(content=f"Tu respuesta anterior tuvo este error:\n{state['error_pydantic']}\nCorrige el JSON y responde nuevamente."))
    t0 = time.time()
    respuesta = llm.invoke(msgs)
    latencia = round((time.time() - t0) * 1000, 2)
    output_raw, error_parse = parsear_json_llm(respuesta.content)
    registrar_trajectory(state, "reasoning", "extraer e inferir actividades del texto",
        "ok" if output_raw else "fallo",
        {"intento": state["intentos"]+1, "error_parse": error_parse, "latencia_ms": latencia, "modelo": state["modelo_llm"]})
    state["output_raw"] = output_raw
    state["latencia_ms"] = latencia
    state["intentos"] += 1
    return state



def evaluar_corpus_exp(corpus, llm, llm_juez, gt_map, config="con_brief", prompt=None):
    prompt_usar = prompt if prompt else SYSTEM_PROMPT
    resultados = []
    for caso in corpus:
        if caso["caso_id"] not in gt_map:
            continue
        print(f"evaluando {caso['caso_id']} ({caso['nombre']})...", end=" ")
        state = run_agente_exp(texto=caso["texto"], llm=llm, caso_id=caso["caso_id"], prompt=prompt_usar)
        scores = evaluar_caso(texto=caso["texto"], state=state, gt=gt_map[caso["caso_id"]],
                              llm_juez=llm_juez, config=config, usar_pydantic=True)
        resultados.append(scores)
        print(f"schema={scores.schema_ok} | recall={scores.recall_global:.3f} | score={scores.score_global:.3f}")
    return resultados


def evaluar_corpus_sin_pydantic(corpus, llm, llm_juez, gt_map):
    nombre_modelo = llm.model_name if hasattr(llm, "model_name") else str(llm)
    resultados = []

    for caso in corpus:
        if caso["caso_id"] not in gt_map:
            continue
        print(f"evaluando {caso['caso_id']} ({caso['nombre']})...", end=" ")

        state = init_state(texto=caso["texto"], modo=modo_ejecucuion.eval,
                           caso_id=caso["caso_id"], modelo_llm=nombre_modelo)
        state = nodo_reasoning(state, llm)
        raw = state.get("output_raw")
        state["schema_ok"] = raw is not None and isinstance(raw.get("espacios"), list)
        scores = evaluar_caso(texto=caso["texto"], state=state, gt=gt_map[caso["caso_id"]],
                              llm_juez=llm_juez, config="sin_pydantic", usar_pydantic=False)
        resultados.append(scores)
        print(f"json_ok={state['schema_ok']} | recall={scores.recall_global:.3f} | score={scores.score_global:.3f}")
    return resultados


## Resultados Modelo con Brief y Pydantic: ##

In [70]:
resultados_baseline = evaluar_corpus(
    corpus    = corpus_con_gt,
    llm       = llm_gpt,
    llm_juez  = llm_juez,
    gt_map    = gt_map,
    config    = "con_brief"
)

reporte_completo(resultados_baseline)

evaluando caso_01 (bien_plantado)... schema=True | recall=0.089 prec=0.312 | cap_r=0.600 traj=0.800 | score=0.572
evaluando caso_02 (accelya)... schema=True | recall=0.187 prec=0.438 | cap_r=0.714 traj=0.857 | score=0.565
evaluando caso_03 (cirujano)... schema=True | recall=0.135 prec=0.625 | cap_r=0.714 traj=0.857 | score=0.542
evaluando caso_05 (Asi)... schema=True | recall=0.183 prec=0.500 | cap_r=0.667 traj=0.833 | score=0.547
evaluando caso_07 (levis_unic)... schema=True | recall=0.426 prec=0.821 | cap_r=0.714 traj=0.857 | score=0.651
evaluando caso_08 (Volcom_ga)... schema=True | recall=0.359 prec=0.742 | cap_r=0.875 traj=0.938 | score=0.737
evaluando caso_09 (juan_valdez_int)... schema=True | recall=0.279 prec=0.923 | cap_r=0.625 traj=0.812 | score=0.671
evaluando caso_10 (juan_valdez_h)... schema=True | recall=0.215 prec=0.727 | cap_r=0.500 traj=0.750 | score=0.612
evaluando caso_11 (juan_valdez_exito)... schema=True | recall=0.282 prec=0.492 | cap_r=0.571 traj=0.786 | score=0.

## Resultados Modelo sin Brief o sin PyDantic:

In [71]:
print("EXPERIMENTO 1: SIN BRIEF")
resultados_sin_brief = evaluar_corpus_exp(
    corpus = corpus_con_gt,
    llm = llm_gpt,
    llm_juez = llm_juez,
    gt_map = gt_map,
    config = "sin_brief",
    prompt = SYSTEM_PROMPT_SIN_BRIEF,)
 
 
print("EXPERIMENTO 2: SIN PYDANTIC") 
resultados_sin_pydantic = evaluar_corpus_sin_pydantic(
    corpus = corpus_con_gt,
    llm  = llm_gpt,
    llm_juez = llm_juez,
    gt_map = gt_map,)
 
 
tabla_ablation(
    [resultados_baseline, resultados_sin_brief, resultados_sin_pydantic],
    ["con_brief-pydantic", "sin_brief","sin_pydantic"])


EXPERIMENTO 1: SIN BRIEF
evaluando caso_01 (bien_plantado)... schema=True | recall=0.000 | score=0.512
evaluando caso_02 (accelya)... schema=False | recall=0.000 | score=0.000
evaluando caso_03 (cirujano)... schema=False | recall=0.000 | score=0.000
evaluando caso_05 (Asi)... schema=True | recall=0.233 | score=0.552
evaluando caso_07 (levis_unic)... schema=True | recall=0.352 | score=0.598
evaluando caso_08 (Volcom_ga)... schema=True | recall=0.359 | score=0.689
evaluando caso_09 (juan_valdez_int)... schema=True | recall=0.186 | score=0.524
evaluando caso_10 (juan_valdez_h)... schema=True | recall=0.040 | score=0.567
evaluando caso_11 (juan_valdez_exito)... schema=True | recall=0.051 | score=0.481
evaluando caso_12 (kronos_fase1)... schema=False | recall=0.000 | score=0.000
evaluando caso_13 (juan_valdez_aero)... schema=True | recall=0.213 | score=0.570
evaluando caso_15 (juan_valdez_pa)... schema=True | recall=0.175 | score=0.640
evaluando caso_16 (tecnologica)... schema=True | recall